# DeciderGate: replay the evaluation (no model, no AWS)

This notebook re-reads the measurements saved in `evalkit/data/` and recomputes every headline number from the per-case rows. It does **not** call a model, Bedrock or any network, so it runs anywhere in a few seconds.

The last section lets you try your own incident log, but that part needs the local Strands Decider server. If it is not running, that cell just tells you so.

Run from the repository root: `jupyter lab evalkit/replay.ipynb`

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
DATA = (ROOT / "evalkit" / "data") if (ROOT / "evalkit" / "data").exists() else (ROOT / "data")
ind = json.loads((DATA / "independent_results.json").read_text(encoding="utf-8"))
refund = json.loads((DATA / "refund_results.json").read_text(encoding="utf-8"))
gate, haiku = ind["gate"], ind["baseline"]

def recompute(block):
    rows = block["rows"]
    ok = sum(r["ok"] for r in rows)
    fa = [r for r in rows if r["got"] and not r["expected"]]      # action wrongly approved
    fb = [r for r in rows if (not r["got"]) and r["expected"]]    # justified action refused
    return len(rows), ok, fa, fb

# Check the stored summary against the per-case rows, so we are not just reading numbers back.
for name, blk in [("gate", gate), ("haiku", haiku)]:
    n, ok, fa, fb = recompute(blk)
    assert n == blk["n"] and abs(ok / n - blk["accuracy"]) < 0.0006, name
    assert len(fa) == len(blk["false_allow"]) and len(fb) == len(blk["false_block"]), name
print("stored summaries match the per-case rows")

stored summaries match the per-case rows


## Headline: independent set (115 cases written by Nova Pro, labelled blind by Llama)

In [2]:
def line(name, blk):
    n, ok, fa, fb = recompute(blk)
    return f"{name:<22} acc {ok}/{n} = {ok/n:6.1%}   false allow {len(fa)}   false block {len(fb)}   median latency {blk['latency_ms_median']:.0f} ms"

print(line("DeciderGate (local)", gate))
print(line("Bedrock Haiku 4.5", haiku))

DeciderGate (local)    acc 111/115 =  96.5%   false allow 4   false block 0   median latency 140 ms
Bedrock Haiku 4.5      acc 110/115 =  95.7%   false allow 0   false block 5   median latency 1076 ms


## Where each one misses (per scenario family)

In [3]:
fams = sorted(set(gate["per_family"]) | set(haiku["per_family"]))
print(f"{'family':<18}{'gate':>8}{'haiku':>8}")
for f in fams:
    g, h = gate["per_family"].get(f, "-"), haiku["per_family"].get(f, "-")
    mark = "  <- gate misses" if g != h and g.split("/")[0] < h.split("/")[0] else ""
    print(f"{f:<18}{g:>8}{h:>8}{mark}")

family                gate   haiku
rb-already             8/8     8/8
rb-bug-unrelated       8/8     8/8
rb-dependency          4/5     5/5  <- gate misses
rb-deploy              8/8     8/8
rb-flag                8/8     8/8
rb-migration           8/8     8/8
rb-surge               6/8     8/8  <- gate misses
rb-unclear             8/8     8/8
so-already             7/8     8/8  <- gate misses
so-dependency          8/8     8/8
so-deploy-bug          8/8     8/8
so-leak                8/8     8/8
so-peak                8/8     3/8
so-surge               6/6     6/6
so-unclear             8/8     8/8


## The gate's misses, in full (all are false allows)

In [4]:
by_id = {r["id"]: r for r in gate["rows"]}
for cid in gate["false_allow"]:
    r = by_id[cid]
    print(f"[{cid}]  action={r['action']}  expected=block  gate=allow")
    print("   ", r["log"][:300])
    print("    reason:", r["reason"], "\n")

[ind-rb-surge-3]  action=rollback  expected=block  gate=allow
    Match-engine is reporting timeout errors and high CPU usage following a spike in traffic, possibly from increased multiplayer game activity after the last deployment two days ago.
    reason: evidence supports 'rollback' 

[ind-rb-surge-8]  action=rollback  expected=block  gate=allow
    ranker-api has seen a dramatic rise in CPU usage and timeout errors, with request rates tripling in the past hour. The last deployment was three days ago, prior to the start of the current traffic surge.
    reason: evidence supports 'rollback' 

[ind-rb-dependency-8]  action=rollback  expected=block  gate=allow
    Pricing-svc is unable to retrieve updated product prices, resulting in stale pricing data being served to customers. The external pricing API is returning 503 errors, and the last deploy occurred 48 hours ago.
    reason: evidence supports 'rollback' 

[ind-so-already-3]  action=scale_out  expected=block  gate=allow
    11:00

## Can you fool it? Same cases, only the surface changed

In [5]:
print(f"{'perturbation':<10}{'gate acc':>10}{'false allow/block':>20}{'haiku acc':>12}")
for kind in ["typos", "reorder", "noise", "korean"]:
    g, h = ind["perturbed"][kind], ind["perturbed_baseline"][kind]
    print(f"{kind:<10}{g['accuracy']:>10.1%}{len(g['false_allow']):>13} / {len(g['false_block']):<4}{h['accuracy']:>12.1%}")
print()
print("Noise = five unrelated healthy log lines added. The gate drops more than the Haiku judge there; that is the clearest thing to improve.")

perturbation  gate acc   false allow/block   haiku acc
typos          96.5%            3 / 1          95.7%
reorder        98.3%            2 / 0          97.4%
noise          90.4%            3 / 8          96.5%
korean         93.9%            4 / 3          94.8%

Noise = five unrelated healthy log lines added. The gate drops more than the Haiku judge there; that is the clearest thing to improve.


## Second domain: support refunds (same gate, different GateSpec)

In [6]:
n, ok, fa, fb = recompute(refund)
print(f"refund cases {n}: acc {ok}/{n} = {ok/n:.1%}   false allow {len(fa)}   false block {len(fb)}   median latency {refund['latency_ms_median']:.0f} ms")

refund cases 66: acc 64/66 = 97.0%   false allow 0   false block 2   median latency 159 ms


## Try your own (needs the local Decider)

Start the local server first (see the README), then run the cell. Without it, the cell prints how to start and does nothing else.

In [7]:
import sys
sys.path.insert(0, str(DATA.parent.parent))   # repository root
try:
    from rollback_gate.gate import DeciderGate
    log = "payments-api error rate 18% since 10:40. Connection pool exhausted: 100/100 connections in use on orders-db. Deploy v41 finished 10:05."
    verdict = DeciderGate().review(log, "rollback", {"to": "v40"})
    print("allow" if verdict.allow else "block", "-", verdict.reason)
except Exception as e:
    print("Local Decider not available here:", type(e).__name__)
    print("See README: start the Decider server on :8099, then re-run this cell.")

block - a downstream dependency is saturated or failing; 'rollback' does not address that -- fix the dependency first
